# 08.11 · The same model in PyTorch (optional)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/11-pytorch/lesson.ipynb)

Read the [README](README.md) first. This notebook rebuilds the 08.8 CNN in PyTorch, line for line, so you can see what Keras was doing for you.

It uses an NVIDIA GPU or an Apple Silicon GPU if you have one and the CPU otherwise. Three epochs take seconds on a GPU and about one to three minutes on a laptop CPU. The first run downloads Fashion-MNIST again (about 30 MB) into `data/raw/`, because PyTorch keeps its own copy.

In [1]:
import time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"                     # NVIDIA GPU (Windows / Linux)
elif torch.backends.mps.is_available():
    device = "mps"                      # Apple Silicon GPU
else:
    device = "cpu"
print("PyTorch", torch.__version__, "· running on", device)

PyTorch 2.14.1 · running on mps


## 1. Data: Dataset and DataLoader

In [2]:
from pathlib import Path

def models_dir():
    """The repo-root models/ folder (git-ignored); created if missing."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "data").is_dir() and (parent / "modules").is_dir():
            folder = parent / "models"
            break
    else:
        folder = Path("models")
    folder.mkdir(exist_ok=True)
    return folder

from torchvision import datasets

def data_dir():
    """data/raw/ in the repo (git-ignored), or a local folder on Colab."""
    return models_dir().parent / "data" / "raw"

train_raw = datasets.FashionMNIST(data_dir(), train=True, download=True)
test_raw = datasets.FashionMNIST(data_dir(), train=False, download=True)
CLASSES = train_raw.classes
print(len(train_raw), "training photos,", len(test_raw), "test photos")

60000 training photos, 10000 test photos


In [3]:
class ProductPhotos(Dataset):
    """Hands out (photo, label) pairs one at a time; DataLoader groups them into batches."""
    def __init__(self, raw):
        self.x = raw.data.float().div(255).unsqueeze(1)   # (N, 1, 28, 28): channels come FIRST in PyTorch
        self.y = raw.targets
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds, test_ds = ProductPhotos(train_raw), ProductPhotos(test_raw)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=512)

xb, yb = next(iter(train_dl))
print("one batch:", tuple(xb.shape), "photos,", tuple(yb.shape), "labels")

one batch: (128, 1, 28, 28) photos, (128,) labels


Keras wanted `(batch, height, width, channels)`. PyTorch wants **`(batch, channels, height, width)`**. Mixing the two up is the most common error when moving between them.

## 2. The model: a class with a `forward` method

In [4]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x):
        return self.head(self.features(x))          # raw scores ("logits"), no softmax

def n_params(model):
    return sum(p.numel() for p in model.parameters())

model = SmallCNN().to(device)
print(model)
print(f"parameters: {n_params(model):,}")

SmallCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=800, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=10, bias=True)
  )
)
parameters: 56,714


56,714, the same as the Keras CNN in 08.8: identical layers, identical weights to learn.

## 3. The training loop you write yourself

`model.fit` in Keras hid these four steps. In PyTorch you write them.

In [5]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()                                    # dropout etc. switch on
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()                        # 1. forget last batch's gradients
        loss = loss_fn(model(xb), yb)                # 2. forward pass and loss
        loss.backward()                              # 3. backpropagation (08.4)
        optimizer.step()                             # 4. gradient-descent update
        total += loss.item() * len(yb)
    return total / len(loader.dataset)

@torch.no_grad()
def accuracy(model, loader):
    model.eval()                                     # dropout etc. switch off
    correct = 0
    for xb, yb in loader:
        correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(loader.dataset)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()                      # applies softmax inside, for numerical safety
start = time.time()
for epoch in range(1, 4):
    loss = train_one_epoch(model, train_dl, optimizer, loss_fn)
    print(f"epoch {epoch}: training loss {loss:.3f}, test accuracy {accuracy(model, test_dl):.3f}")
print(f"{time.time() - start:.0f} s on {device}")

epoch 1: training loss 0.698, test accuracy 0.822


epoch 2: training loss 0.424, test accuracy 0.852


epoch 3: training loss 0.373, test accuracy 0.867
4 s on mps


(We print test accuracy each epoch only to keep the lesson short. In a real project use a validation set, as in 04.1, and look at the test set once.)

## 4. Predicting one photo

In [6]:
model.eval()
with torch.no_grad():
    logits = model(test_ds.x[:1].to(device))
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
print("true:", CLASSES[test_ds.y[0]])
for k in np.argsort(probs)[::-1][:3]:
    print(f"  {CLASSES[k]:<12} {probs[k]:.3f}")

true: Ankle boot
  Ankle boot   0.861
  Sneaker      0.106
  Sandal       0.029


## 5. Saving

PyTorch saves the learned numbers (the **state dict**); to load them you need the class definition too.

In [7]:
path = models_dir() / "fashion_cnn_pytorch.pt"
torch.save(model.state_dict(), path)
print("saved", path.name, f"({path.stat().st_size / 1e3:.0f} kB)")

saved fashion_cnn_pytorch.pt (231 kB)


## Quiz answers

1. Keras: `(batch, height, width, channels)`; PyTorch: `(batch, channels, height, width)`.
2. Zero the old gradients, forward pass and loss, `loss.backward()`, `optimizer.step()`.
3. Without it, PyTorch adds each batch's gradients to the previous ones, so updates become wrong.
4. It switches layers like dropout and batch normalisation into prediction mode; forgetting it gives noisy, worse predictions.
5. `nn.CrossEntropyLoss` applies softmax itself, so the model outputs raw scores (logits).

**Next:** the [module project](../project/).